In [53]:
import pandas as pd
import warnings
import numpy as np
from typing import Literal
warnings.filterwarnings('ignore')

In [54]:
file_path = r"FECS\419725395ExportStructureFEC20250831.txt"
# file_path = r"FECS\419725395ExportStructureFEC20250930.txt"
liste_encoding = ["utf-8","MacRoman"]
try: 
    df = pd.read_csv(file_path, sep='\t', encoding= liste_encoding[0], engine='python')
    if df.shape[1] == 1:
          df = pd.read_csv(file_path, sep='|', encoding= liste_encoding[0], engine='python')
except UnicodeDecodeError:
        df = pd.read_csv(file_path, sep='\t', encoding= liste_encoding[1], engine='python')
        if df.shape[1] == 1:
            df = pd.read_csv(file_path, sep='|', encoding= liste_encoding[1], engine='python')
df.head()

,JournalCode,JournalLib,EcritureNum,EcritureDate,CompteNum,CompteLib,CompAuxNum,CompAuxLib,PieceRef,PieceDate,EcritureLib,Debit,Credit,EcritureLet,DateLet,ValidDate,Montantdevise,Idevise
0,AN,A-NOUVEAU,P0,20250101,411,Collectif client,C00006,A2M,EMI23110019,20250101,08/11/2023 -VT- A2M,"254,40","0,00",NaN,NaN,NaN,"0,00",NaN
1,AN,A-NOUVEAU,P0,20250101,411,Collectif client,C00024,ADLER S.A. MOUSSY LE NEUF,EMI24120013,20250101,06/12/2024 -VT- ADLER S.A. MOUSSY LE NEUF,"6,78","0,00",B,20250124.0,NaN,"0,00",NaN
2,AN,A-NOUVEAU,P0,20250101,411,Collectif client,C00025,GROUPE ADP ROISSY CDG,EMI24120070,20250101,18/12/2024 -VT- GROUPE ADP ROISSY CDG,"2407,88","0,00",H,20250214.0,NaN,"0,00",NaN
3,AN,A-NOUVEAU,P0,20250101,411,Collectif client,C00025,GROUPE ADP ROISSY CDG,EMI24120071,20250101,18/12/2024 -VT- GROUPE ADP ROISSY CDG,"2407,88","0,00",H,20250214.0,NaN,"0,00",NaN
4,AN,A-NOUVEAU,P0,20250101,411,Collectif client,C00025,GROUPE ADP ROISSY CDG,EMI24120077,20250101,30/12/2024 -VT- GROUPE ADP ROISSY CDG,"246,00","0,00",G,20250220.0,NaN,"0,00",NaN


In [55]:
df["EcritureDate"] = pd.to_datetime(df["EcritureDate"],format='%Y%m%d', errors='coerce')
df['Année'] = df["EcritureDate"].dt.year
df['Mois'] = df["EcritureDate"].dt.month_name(locale='French')

def reorder_months(df):
    mois_en_ordre = [
        "Janvier", "Février", "Mars", "Avril", "Mai", "Juin",
        "Juillet", "Août", "Septembre", "Octobre", "Novembre", "Décembre"
    ]

    # Handle empty DataFrame
    if df.empty or 'Année' not in df or 'Mois' not in df:
        return []

    # Get all unique years in ascending order
    years = sorted(df['Année'].dropna().unique())
    if len(years) == 0:
        return []

    # Get data for the oldest year
    oldest_year = years[0]
    year_data = df[df['Année'] == oldest_year].copy()

    # Keep only months present in dataset
    mois_present = [m for m in mois_en_ordre if m in df['Mois'].unique()]
    if not mois_present:
        return []

    # Convert to categorical for correct ordering
    year_data['Mois'] = pd.Categorical(
        year_data['Mois'], categories=mois_en_ordre, ordered=True
    )

    # Get the first valid month (minimum by order)
    first_month = year_data['Mois'].min()
    if pd.isna(first_month):
        return mois_present  # No valid months for that year

    # Rotate the list so it starts with the first month found
    if first_month in mois_present:
        start_idx = mois_present.index(first_month)
        reordered_months = mois_present[start_idx:] + mois_present[:start_idx]
    else:
        reordered_months = mois_present  # fallback if mismatch

    return reordered_months

# Replace the original mois_en_ordre assignment with:
mois_en_ordre = reorder_months(df)

df["Credit"] = pd.to_numeric(df["Credit"].str.replace(',', '.', regex=True), errors='coerce').fillna(0)
df["Debit"] = pd.to_numeric(df["Debit"].str.replace(',', '.', regex=True), errors='coerce').fillna(0)

def custom_round(value):
    # If the decimal part is greater than or equal to 0.5, round up
    if value - np.floor(value) >= 0.5:
        return int(np.ceil(value))  # Round up and convert to int
    else:
        return int(np.floor(value))  # Round down and convert to int


In [56]:
mois_en_ordre

['Janvier', 'Février', 'Mars', 'Avril', 'Mai', 'Juin', 'Juillet', 'Août']

In [57]:
df["Mois"].unique()

array(['Janvier', 'Février', 'Mars', 'Avril', 'Mai', 'Juin', 'Juillet',
       'Août'], dtype=object)

## Cut-off Ventes

In [58]:
def cut_off(df,type:Literal["achats","ventes","avoirs d'achat","avoirs de vente"],intervalle_periode=None,compte_num_column = "CompteNum",mois_en_ordre=None,num_months = 2,top_max=20,seuil=3000):

    """
    Select and calculate cut-off for given type of transactions.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the transactions.
    type : Literal["achats","ventes","avoirs d'achat","avoirs de vente"]
        The type of transactions to select and calculate cut-off for.
    periode : str, optional
        The period for which to select and calculate cut-off. If None, defaults to the last two months of the year.
    compte_num_column : str, optional
        The column containing the account numbers. Defaults to "CompteNum".
    mois_en_ordre : list, optional
        The list of months in order. Defaults to None.
    top_max : int, optional
        The maximum number of rows to select for the top selection. Defaults to 20.

    Returns
    -------
    tuple
        A tuple containing the title of the selection, the DataFrame of the selection, and an error message if there are no data for the given period.
    """
    
    compte_num,liste_journal_code = ('401',["AC","HA"]) if type in ["achats","avoirs d'achat"] else ('411',["VE","VT","VTE"])
    colonne_de_calcul = "Credit" if type in ["achats","avoirs de vente"] else "Debit"
    
    periode = None
    if mois_en_ordre is not None:
        if intervalle_periode is not None:
            from_,to_ = intervalle_periode
            periode = mois_en_ordre[from_:to_+1]
        else:
            periode = mois_en_ordre[:num_months] if type in ["achats","avoirs de vente"] else mois_en_ordre[-num_months:]

    data = df[(df["JournalCode"].str.strip().isin(liste_journal_code)) & (df[compte_num_column].astype(str).str.startswith(compte_num))].loc[df['Mois'].isin((periode))]
    periode_annee = f"{'-'.join([mois for mois in periode])} {df['Année'].unique()[-1]}"

    titles =   {"titre_wb" : f'Cut-off des {type} à la clôture ({periode_annee.split("-")[-1].split(" ")[0]})',
                "titre_selection_cut_off": f'Sélection cut-off des {type} sur la période {periode_annee}',
                "titre_document_apportes": 'Documents apportés par le client'
                }
    
    empty_df = pd.DataFrame(columns=["Date d'écriture comptable", 'Compte', 'Libellé', 'Debit', 'Credit'])
    error = f"Il n'y a pas de données pour les {type} durant la période {periode_annee}"    

    if data.shape[0] != 0 : 
        classement = data[["JournalCode",'EcritureDate',compte_num_column,'PieceRef','EcritureLib',colonne_de_calcul]].sort_values(by=colonne_de_calcul,ascending=False)
        
        if int(classement[colonne_de_calcul].unique()[0]) == 0:
                return titles,empty_df,error
        
        classement[colonne_de_calcul] = classement[colonne_de_calcul].apply(custom_round)
        classement=classement[~(classement[colonne_de_calcul] == 0)]
        classement.columns = ["Code","Date d'écriture comptable", 'Compte', 'PieceRef','Libellé', colonne_de_calcul]
        classement["Date d'écriture comptable"] = classement["Date d'écriture comptable"].dt.strftime('%d/%m/%Y')

        periode_annee = f"{'-'.join([mois for mois in periode])} {data['Année'].unique()[0]}"

        top = classement.head(top_max)
        top = top[top[colonne_de_calcul]>=seuil] if type in ["avoirs d'achat","avoirs de vente"] else top
        if top.shape[0] == 0:
            error = f"Il n'y a pas d'{type} durant la période {periode_annee} au dessus de {seuil}€"
            return titles,empty_df,error
        
        top["Sélection"] = top[colonne_de_calcul]
        total_colonne_calcul = classement[colonne_de_calcul].sum()
        total_selection = top["Sélection"].sum()

        pourcentage_de_selectionnés = f"{round((total_selection/total_colonne_calcul)*100)} %" if total_colonne_calcul not in [None,0,0.0] and total_selection not in [None,0,0.0] else None
        bottom = pd.DataFrame([
                                {   "Code":'',
                                    "Date d'écriture comptable" : "",
                                    "Compte" : "",
                                    "PieceRef":'',
                                    "Libellé" : f"Total {type} {periode_annee} (hors avoirs)" if type in ["achats","ventes"] else f"Total {type} {periode_annee}",
                                    colonne_de_calcul: total_colonne_calcul,
                                    "Sélection" : total_selection
                                },
                                {   "Code":'',
                                    "Date d'écriture comptable" : "",
                                    "Compte" : "",
                                    "PieceRef":'',
                                    "Libellé" : "",
                                    colonne_de_calcul : "Total sélectionné",
                                    "Sélection" : total_selection
                                },
                                {   "Code":'',
                                    "Date d'écriture comptable" : "",
                                    "Compte" : "",
                                    "PieceRef":'',
                                    "Libellé" : "",
                                    colonne_de_calcul: f"% {type} sélectionné(e)s",
                                    "Sélection" : pourcentage_de_selectionnés
                                },
                                {   "Code":'',
                                    "Date d'écriture comptable" : "",
                                    "Compte" : "",
                                    "PieceRef":'',
                                    "Libellé" : "",
                                    colonne_de_calcul : "Nombre de factures totales",
                                    "Sélection" : classement.shape[0]
                                },
                                {   "Code":'',
                                    "Date d'écriture comptable" : "",
                                    "Compte" : "",
                                    "PieceRef":'',
                                    "Libellé" : "",
                                    colonne_de_calcul : "Nombre de factures sélectionnées",
                                    "Sélection" : len(top)
                                }
                            ])

        top_5_final = pd.concat([top,bottom],axis=0)
        top_5_final = top_5_final.reset_index(drop=True)

        titles =   {"titre_wb" : f'Cut-off des {type} à la clôture ({periode_annee.split("-")[-1].split(" ")[0]})',
                    "titre_selection_cut_off": f'Sélection cut-off des {type} sur la période {periode_annee}',
                    "titre_document_apportes": 'Documents apportés par le client'
                    }
        return titles,top_5_final,None

    return titles,empty_df,error

In [59]:
type_list=["achats","ventes","avoirs d'achat","avoirs de vente"]
titre,selection_cut_off,error = cut_off(df,compte_num_column = "CompteNum",type = type_list[1],mois_en_ordre = mois_en_ordre,top_max=20,num_months=4)
print(titre["titre_selection_cut_off"])
selection_cut_off

Sélection cut-off des ventes sur la période Mai-Juin-Juillet-Août 2025


,Code,Date d'écriture comptable,Compte,PieceRef,Libellé,Debit,Sélection
0,VT,01/07/2025,411,EMI25070001,LAKALUX SAS,27686,27686
1,VT,16/06/2025,411,EMI25060048,RATP BAGNEUX HALL A3,25942,25942
2,VT,26/05/2025,411,EMI25050074,LIGNE 18,18705,18705
3,VT,19/08/2025,411,EMI25080055,MCFE RATP MASSY,16978,16978
4,VT,24/07/2025,411,EMI25070136,LIGNE 18,13545,13545
5,VT,09/07/2025,411,EMI25070078,MOVIANTO (CSP) MOUSSY LE,12529,12529
6,VT,24/06/2025,411,EMI25060086,GROUPE VAUBAN ST OUEN L'A,12108,12108
7,VT,01/08/2025,411,EMI25080028,SYSTRONIC,11956,11956
8,VT,22/05/2025,411,EMI25050058,MAN TRUCKS BUS ST MARD,11880,11880
9,VT,31/05/2025,411,EMI25050105,VINYL DE PARIS CHELLES,11155,11155


In [60]:
min([2024,2025])

2024